# Day 03: t-SNE (van der Maaten & Hinton, JMLR 2008)
### Sổ Tay Thực Hành From Scratch: `tsne_scratch.ipynb`

> **Bài báo gốc:** *Visualizing Data using t-SNE*  
> **Tác giả:** Laurens van der Maaten & Geoffrey Hinton  
> **Năm công bố:** 2008 — Tạp chí *Journal of Machine Learning Research (JMLR)*  
> **Tài liệu tham khảo:** [`resources/Day 3 Visualizing Data using t-SNE.pdf`](./resources/Day%203%20Visualizing%20Data%20using%20t-SNE.pdf)

---

## 🎯 Mục Tiêu Cốt Lõi Của Notebook
1. **Giải thích chi tiết cho người mới bắt đầu:** Hiểu rõ bản chất hình học của Lời nguyền số chiều, tại sao diện tích 2D không đủ chứa láng giềng nhiều chiều dẫn tới "Vấn nạn tập trung quá mức" (**The Crowding Problem**), và cách phân phối t-Student đuôi dày (Heavy-tailed Cauchy) giải quyết triệt để vấn đề này.
2. **Tự lập trình từ con số 0 (From Scratch thuần NumPy):** Xây dựng thuật toán tính ma trận khoảng cách vector hóa, tìm kiếm nhị phân phương sai $\sigma_i$ thích ứng theo Perplexity, tính gradient giải tích đối xứng và tối ưu hóa bằng Momentum + Early Exaggeration.
3. **Thực nghiệm đối chuẩn giá trị học thuật:** Tái hiện và kiểm chứng định lượng sự tiến hóa:
   - **Linear PCA (1901):** Chiếu tuyến tính phẳng, làm đè dẹp các cụm.
   - **Gaussian SNE (2002):** Dùng Gaussian ở 2D, các cụm bị nén chặt vào tâm (Crowding Problem).
   - **t-SNE (2008):** Dùng t-Student ở 2D, các cụm tự do bung tỏa và tách biệt rõ rệt.

## 🏛️ Bối Cảnh Lịch Sử: Trước và Sau Khi Bài Báo Ra Đời

| Tiêu Chí So Sánh | Linear PCA (1901) | Gaussian SNE (2002) | t-SNE (Hinton 2008) |
|---|---|---|---|
| **Cách tiếp cận** | Hình học tuyến tính (Phương sai cực đại) | Khớp phân phối xác suất (Gaussian-Gaussian) | **Khớp xác suất (Gaussian - t-Student)** |
| **Phân phối không gian 2D** | Chiếu trực giao | Phân phối Gaussian (Đuôi mỏng $\exp(-d^2)$) | **Phân phối t-Student (Đuôi dày $(1 + d^2)^{-1}$)** |
| **Hiện tượng Crowding** | N/A | **Nghiêm trọng (các cụm bị ép chặt vào tâm)** | **Hóa giải hoàn toàn (tách thành các đảo độc lập)** |
| **Độ nhạy outliers** | Rất cao | Cao do xác suất có điều kiện bất đối xứng | **Thấp nhờ Symmetric SNE ($p_{ij} = (p_{j|i}+p_{i|j})/2N$)** |
| **Khả năng hiển thị cụm** | Kém (chồng lấn) | Trung bình (bị dồn ứ) | **Xuất sắc, chuẩn mực vàng trực quan hóa** |

In [ ]:
import time
import numpy as np

print("[✓] Đã khởi tạo môi trường tính toán số học thuần NumPy!")

## PHẦN 1: CÁC KHỐI TOÁN HỌC KHÔNG GIAN CAO (P-MATRIX & BINARY SEARCH)

Xác suất có điều kiện điểm $x_i$ chọn $x_j$ làm láng giềng theo phân phối chuẩn Gaussian:
$$p_{j|i} = \frac{\exp\left(-\frac{\|x_i - x_j\|^2}{2\sigma_i^2}\right)}{\sum_{k \neq i} \exp\left(-\frac{\|x_i - x_k\|^2}{2\sigma_i^2}\right)}$$

Phương sai $\sigma_i$ được tìm kiếm bằng thuật toán **tìm kiếm nhị phân (Binary Search)** sao cho Entropy Shannon khớp với tham số Perplexity do người dùng thiết lập:
$$\text{Perp}(P_i) = 2^{H(P_i)}, \quad H(P_i) = -\sum_{j} p_{j|i} \log_2 p_{j|i}$$

In [ ]:
def compute_pairwise_distances(X):
    """Tính ma trận khoảng cách bình phương Euclidean: D_ij = ||x_i - x_j||^2"""
    sum_X = np.sum(np.square(X), axis=1)
    D = np.add(np.add(-2 * np.dot(X, X.T), sum_X).T, sum_X)
    np.fill_diagonal(D, 0.0)
    return np.maximum(D, 0.0)

def compute_h_and_p(diff_row, beta):
    P = np.exp(-diff_row * beta)
    sum_P = np.sum(P)
    if sum_P < 1e-15:
        sum_P = 1e-15
    H = np.log(sum_P) + beta * np.sum(diff_row * P) / sum_P
    return H, P / sum_P

def binary_search_perplexity(distances, target_perplexity=30.0, tol=1e-5, max_iter=50):
    N = distances.shape[0]
    P = np.zeros((N, N), dtype=np.float64)
    target_H = np.log(target_perplexity)

    for i in range(N):
        diff_row = np.delete(distances[i], i)
        beta_min, beta_max, beta = -np.inf, np.inf, 1.0
        H, this_P = compute_h_and_p(diff_row, beta)
        h_diff = H - target_H
        tries = 0
        while np.abs(h_diff) > tol and tries < max_iter:
            if h_diff > 0:
                beta_min = beta
                beta = beta * 2.0 if np.isinf(beta_max) else (beta + beta_max) / 2.0
            else:
                beta_max = beta
                beta = beta / 2.0 if np.isinf(beta_min) else (beta + beta_min) / 2.0
            H, this_P = compute_h_and_p(diff_row, beta)
            h_diff = H - target_H
            tries += 1
        P[i, np.delete(np.arange(N), i)] = this_P
    return P

def compute_joint_probabilities(X, target_perplexity=30.0):
    D = compute_pairwise_distances(X)
    cond_P = binary_search_perplexity(D, target_perplexity)
    N = X.shape[0]
    P = (cond_P + cond_P.T) / (2.0 * N)
    np.fill_diagonal(P, 0.0)
    P = np.maximum(P, 1e-12)
    return P / np.sum(P)

print("[✓] Đã khởi tạo các hàm tính xác suất không gian cao P!")

## PHẦN 2: PHÂN PHỐI KHÔNG GIAN 2D & ĐẠO HÀM GRADIENT

1. **Phân phối t-Student (Cauchy 1 bậc tự do - t-SNE 2008):**
   $$q_{ij} = \frac{(1 + \|y_i - y_j\|^2)^{-1}}{\sum_{k} \sum_{l \neq k} (1 + \|y_k - y_l\|^2)^{-1}}$$
   - Đạo hàm giải tích: $\frac{\partial C}{\partial y_i} = 4 \sum_j (p_{ij} - q_{ij})(y_i - y_j)(1 + \|y_i - y_j\|^2)^{-1}$

2. **Phân phối Gaussian (SNE nguyên bản 2002):**
   $$q_{ij}^{\text{Gauss}} = \frac{\exp(-\|y_i - y_j\|^2)}{\sum_{k} \sum_{l \neq k} \exp(-\|y_k - y_l\|^2)}$$
   - Đạo hàm: $\frac{\partial C}{\partial y_i} = 4 \sum_j (p_{ij} - q_{ij})(y_i - y_j)$ (không có số hạng giảm chấn nghịch đảo bình phương $\implies$ gây ra Crowding Problem).

In [ ]:
def compute_q_student(Y):
    sum_Y = np.sum(np.square(Y), axis=1)
    dist_sq = np.add(np.add(-2 * np.dot(Y, Y.T), sum_Y).T, sum_Y)
    np.fill_diagonal(dist_sq, 0.0)
    num = 1.0 / (1.0 + dist_sq)
    np.fill_diagonal(num, 0.0)
    sum_num = max(np.sum(num), 1e-15)
    return np.maximum(num / sum_num, 1e-12), num

def compute_q_gaussian(Y):
    sum_Y = np.sum(np.square(Y), axis=1)
    dist_sq = np.add(np.add(-2 * np.dot(Y, Y.T), sum_Y).T, sum_Y)
    np.fill_diagonal(dist_sq, 0.0)
    num = np.exp(-dist_sq)
    np.fill_diagonal(num, 0.0)
    sum_num = max(np.sum(num), 1e-15)
    return np.maximum(num / sum_num, 1e-12), num

def compute_gradient_tsne(P, Q, num, Y):
    L = (P - Q) * num
    return 4.0 * (np.sum(L, axis=1, keepdims=True) * Y - np.dot(L, Y))

def compute_gradient_gaussian_sne(P, Q, Y):
    L = (P - Q)
    return 4.0 * (np.sum(L, axis=1, keepdims=True) * Y - np.dot(L, Y))

print("[✓] Đã khởi tạo các hàm phân phối và đạo hàm không gian thấp Q!")

## PHẦN 3: LỚP THUẬT TOÁN t-SNE & GAUSSIAN SNE (FROM SCRATCH)

In [ ]:
class TSNE_Scratch:
    def __init__(self, n_components=2, perplexity=30.0, learning_rate=150.0,
                 n_iter=500, early_exaggeration_iter=150, exaggeration_factor=4.0,
                 random_state=42):
        self.n_components = n_components
        self.perplexity = perplexity
        self.learning_rate = learning_rate
        self.n_iter = n_iter
        self.early_exaggeration_iter = early_exaggeration_iter
        self.exaggeration_factor = exaggeration_factor
        self.random_state = random_state

    def fit_transform(self, X):
        np.random.seed(self.random_state)
        N = X.shape[0]
        P = compute_joint_probabilities(X, self.perplexity)
        P_ex = P * self.exaggeration_factor
        Y = np.random.randn(N, self.n_components) * 1e-4
        Y_step = np.zeros_like(Y)

        for it in range(1, self.n_iter + 1):
            curr_P = P_ex if it <= self.early_exaggeration_iter else P
            Q, num = compute_q_student(Y)
            dY = compute_gradient_tsne(curr_P, Q, num, Y)
            momentum = 0.5 if it < self.early_exaggeration_iter else 0.8
            Y_step = momentum * Y_step - self.learning_rate * dY
            Y = Y + Y_step
            Y = Y - np.mean(Y, axis=0)
        return Y


class Gaussian_SNE_Scratch:
    def __init__(self, n_components=2, perplexity=30.0, learning_rate=150.0,
                 n_iter=500, early_exaggeration_iter=150, exaggeration_factor=4.0,
                 random_state=42):
        self.n_components = n_components
        self.perplexity = perplexity
        self.learning_rate = learning_rate
        self.n_iter = n_iter
        self.early_exaggeration_iter = early_exaggeration_iter
        self.exaggeration_factor = exaggeration_factor
        self.random_state = random_state

    def fit_transform(self, X):
        np.random.seed(self.random_state)
        N = X.shape[0]
        P = compute_joint_probabilities(X, self.perplexity)
        P_ex = P * self.exaggeration_factor
        Y = np.random.randn(N, self.n_components) * 1e-4
        Y_step = np.zeros_like(Y)

        for it in range(1, self.n_iter + 1):
            curr_P = P_ex if it <= self.early_exaggeration_iter else P
            Q, num = compute_q_gaussian(Y)
            dY = compute_gradient_gaussian_sne(curr_P, Q, Y)
            momentum = 0.5 if it < self.early_exaggeration_iter else 0.8
            Y_step = momentum * Y_step - self.learning_rate * dY
            Y = Y + Y_step
            Y = Y - np.mean(Y, axis=0)
        return Y

print("[✓] Đã khởi tạo thành công TSNE_Scratch và Gaussian_SNE_Scratch!")

## PHẦN 4: THỰC NGHIỆM ĐỐI CHUẨN GIÁ TRỊ HỌC THUẬT (ACADEMIC VALUE BENCHMARK)

Chúng ta tạo một tập dữ liệu đa cụm trong không gian 12 chiều để so sánh:
1. **Linear PCA (1901)**
2. **Gaussian SNE (2002)**: Khảo sát hiện tượng Crowding Problem.
3. **t-SNE (Hinton 2008)**: Khảo sát sự tách cụm với phân phối t-Student.

In [ ]:
# Tạo dữ liệu đa cụm trong không gian 12 chiều
np.random.seed(42)
n_clusters = 3
n_samples = 40
n_features = 12

X_list, y_list = [], []
for c in range(n_clusters):
    center = np.random.uniform(-8, 8, size=(n_features,))
    X_list.append(center + np.random.randn(n_samples, n_features) * 1.0)
    y_list.append(np.full((n_samples,), c))

X = np.vstack(X_list)
y = np.concatenate(y_list)
print(f"[*] Dữ liệu thử nghiệm: {X.shape[0]} mẫu x {X.shape[1]} chiều, {n_clusters} cụm.")

In [ ]:
# Hàm đo lường Tỷ số Tách Cụm (Separation Ratio = Inter / Intra Distance)
def separation_ratio(Y, labels, k=3):
    centers = [np.mean(Y[labels == c], axis=0) for c in range(k)]
    intras = [np.mean(np.linalg.norm(Y[labels == c] - centers[c], axis=1)) for c in range(k)]
    inters = [np.linalg.norm(centers[i] - centers[j]) for i in range(k) for j in range(i + 1, k)]
    return np.mean(inters) / (np.mean(intras) + 1e-8)

# 1. Linear PCA (1901)
t0 = time.time()
X_c = X - np.mean(X, axis=0)
_, _, Vt = np.linalg.svd(X_c, full_matrices=False)
Y_pca = np.dot(X_c, Vt[:2].T)
pca_time = time.time() - t0
pca_sep = separation_ratio(Y_pca, y)

# 2. Gaussian SNE (2002)
t0 = time.time()
sne = Gaussian_SNE_Scratch(perplexity=15.0, learning_rate=150.0, n_iter=400, random_state=42)
Y_sne = sne.fit_transform(X)
sne_time = time.time() - t0
sne_sep = separation_ratio(Y_sne, y)

# 3. t-SNE (Hinton 2008)
t0 = time.time()
tsne = TSNE_Scratch(perplexity=15.0, learning_rate=150.0, n_iter=400, random_state=42)
Y_tsne = tsne.fit_transform(X)
tsne_time = time.time() - t0
tsne_sep = separation_ratio(Y_tsne, y)

# BẢNG ĐỐI CHUẨN ĐỊNH LƯỢNG
print("=" * 88)
print(f"{'Mô Hình / Giải Thuật':<26} | {'Phân Phối 2D':<14} | {'Tỷ Số Tách Cụm (Separation)':<30} | Thời Gian")
print("=" * 88)
print(f"{'1. Linear PCA (1901)':<26} | {'Tuyến tính':<14} | {pca_sep:12.2f} (Các cụm bị đè dẹp)     | {pca_time:6.2f}s")
print(f"{'2. Gaussian SNE (2002)':<26} | {'Gaussian':<14} | {sne_sep:12.2f} (Bị kẹt Crowding)       | {sne_time:6.2f}s")
print(f"{'3. t-SNE (Hinton 2008)':<26} | {'t-Student':<14} | {tsne_sep:12.2f} (⭐ Tách biệt hoàn hảo!) | {tsne_time:6.2f}s")
print("=" * 88)

## PHẦN 5: TRỰC QUAN HÓA SO SÁNH BIỂU ĐỒ PHÂN TÁN ASCII TRÊN TERMINAL

In [ ]:
def render_ascii(Y, labels, title, w=50, h=16):
    grid = [[' ' for _ in range(w)] for _ in range(h)]
    xc, yc = Y[:, 0], Y[:, 1]
    xmin, xmax = np.min(xc), np.max(xc)
    ymin, ymax = np.min(yc), np.max(yc)
    syms = ['#', '*', '@']
    for i in range(len(Y)):
        gx = int((xc[i] - xmin) / (xmax - xmin + 1e-9) * (w - 1))
        gy = h - 1 - int((yc[i] - ymin) / (ymax - ymin + 1e-9) * (h - 1))
        grid[gy][gx] = syms[labels[i] % len(syms)]
    print(f"\n[{title}]")
    print("+" + "-" * w + "+")
    for r in grid:
        print("|" + "".join(r) + "|")
    print("+" + "-" * w + "+")

render_ascii(Y_sne, y, title="Gaussian SNE (2002) - Crowding Problem: Các cụm bị nén vào tâm")
render_ascii(Y_tsne, y, title="t-SNE (Hinton 2008) - Phân phối t-Student: Các cụm tách biệt rõ rệt")

## 🎓 KẾT LUẬN & ĐÚC KẾT HỌC THUẬT DÀNH CHO NGƯỜI MỚI BẮT ĐẦU

1. **Bản chất của Crowding Problem:**
   - Khi chuyển từ không gian nhiều chiều xuống 2D, thể tích mặt cầu suy giảm đột ngột khiến không gian hiển thị bị co hẹp.
   - Nếu dùng phân phối Gaussian ở 2D, chi phí phạt khoảng cách quá nặng khiến các điểm bị dồn ứ chặt vào tâm biểu đồ.
2. **Sức mạnh của Phân phối t-Student (1 bậc tự do / Cauchy):**
   - Đuôi dày của t-Student cho phép các điểm thuộc các cụm khác nhau đứng cách xa nhau mà không bị phạt.
   - Điều này tạo ra lực đẩy tự nhiên giữa các cụm, giúp t-SNE tạo ra các "hòn đảo" phân cụm tách biệt rõ nét.